# Owner Name Entity Resolution Pipeline

Cluster similar owner names into a single canonical owner while **minimizing false positives**.

| Item | Value |
|------|-------|
| Dataset | `temp_unique_corporate_names` (~349,413 unique names) |
| Embedding model | Qwen Embedding (`Qwen/Qwen3-Embedding-*`) |
| Search engine | FAISS (`IndexFlatIP`) |
| Clustering | Union-Find (Disjoint Set) |

**Pipeline:** Cleaning -> Typo Normalization -> Canonical Name -> Embedding -> FAISS Search -> Candidate Pairs -> Features -> Rule Engine -> Union-Find -> Cluster + Canonical Selection -> Evaluation -> Incremental.

> Notebook is code-only scaffold. Cells are NOT executed. Run top-to-bottom after installing deps.

## Step 0 : Setup & Configuration

In [1]:
# Step 0.1 : Install dependencies (run once, uncomment)
# !pip install pandas numpy faiss-cpu sentence-transformers rapidfuzz python-Levenshtein tqdm unidecode
# For GPU embedding + search use: faiss-gpu and a CUDA-enabled torch

In [1]:
# Step 0.2 : Imports
import os
import re
import json
import unicodedata
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Fuzzy string metrics
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein

# Embedding + search (imported lazily in their phases to keep this cell light)
# from sentence_transformers import SentenceTransformer
# import faiss

In [2]:
# Step 0.3 : Global config / paths
CONFIG = {
    # ---- IO ----
    "input_file":        "owner_name_enrichment_needs_final_training_owners_dataset",   # single column, header = owner_name_cleaned
    "input_col":         "owner_name_cleaned",
    "artifacts_dir":     "artifacts",

    # ---- Embedding ----
    "model_name":        "Qwen/Qwen3-Embedding-8B",  # 4096-dim -> matches plan example (349413, 4096)
    "batch_size":        256,
    "normalize":         True,          # L2-normalize embeddings so IP == cosine

    # ---- FAISS ----
    "top_k":             100,           # neighbors per name

    # ---- Rule engine thresholds (tune in Phase 8) ----
    "rules": {
        "r1_cosine":     0.97, "r1_token_set": 95, "r1_rapidfuzz": 90,
        "r2_lev":        2,    "r2_cosine":    0.94,
        "min_cosine_gate": 0.85,   # coarse pruning gate before feature calc
        "core_sim_min":  85,       # distinguishing-part similarity floor (reject (b))
        "near_exact_rf": 95, "near_exact_cos": 0.97,   # full-string near-identity bypass (concat/spacing)
        "r3_cosine":     0.95, "r3_core_sim": 90,      # abbreviation-expansion merge (INV->INVEST)
    },
}

os.makedirs(CONFIG["artifacts_dir"], exist_ok=True)

def art(name):
    """Path helper inside artifacts dir."""
    return os.path.join(CONFIG["artifacts_dir"], name)

CONFIG

{'input_file': 'owner_name_enrichment_needs_final_training_owners_dataset',
 'input_col': 'owner_name_cleaned',
 'artifacts_dir': 'artifacts',
 'model_name': 'Qwen/Qwen3-Embedding-8B',
 'batch_size': 256,
 'normalize': True,
 'top_k': 100,
 'rules': {'r1_cosine': 0.97,
  'r1_token_set': 95,
  'r1_rapidfuzz': 90,
  'r2_lev': 2,
  'r2_cosine': 0.94,
  'min_cosine_gate': 0.85,
  'core_sim_min': 85,
  'near_exact_rf': 95,
  'near_exact_cos': 0.97,
  'r3_cosine': 0.95,
  'r3_core_sim': 90}}

## Phase 1 : Data Cleaning
Standardize names before embedding. Uppercase, collapse spaces, replace punctuation, unicode-normalize, trim.
Cleaning must **never remove meaningful words**.

In [4]:
# Step 1.1 : Load raw data
df = pd.read_csv(CONFIG["input_file"])
df = df.rename(columns={CONFIG["input_col"]: "original_name"})
df["original_name"] = df["original_name"].astype(str)

print("rows:", len(df))
df.head(10)

rows: 378716


,original_name
0,EMINENT RESEARCH & ADVISORY SERVICE
1,S R B C & COMPANY LLP
2,NAKODA NUTRIMENTS & BIOSCI PRIVATE LIMITED
3,WIPRO PARI ENGINEERING & SERVICES
4,ESQUIRE HEALTHCARE & LGTS PRIVATE LIMITED
5,TASTE & TRENDS HOSPITALITY PRIVATE LIMITED
6,JAY MALHAR TOURS & TRAVELS
7,KRISHNA TOURS & TRAVELS
8,OM TOURS & TRAVELS
9,PAVAN TOURS & TRAVELS


In [29]:
# Step 1.2 : Cleaning function
# & -> AND ; other punctuation ( , . - / ( ) etc ) -> space ; unicode accents stripped
_PUNCT_TO_SPACE = re.compile(r"[^A-Z0-9&\s]")
_MULTISPACE     = re.compile(r"\s+")

def clean_name(name: str) -> str:
    if not isinstance(name, str):
        return ""
    # unicode normalize: E' -> E (strip accents)
    s = unicodedata.normalize("NFKD", name)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    # uppercase
    s = s.upper()
    # ampersand -> AND (spaced so it becomes its own token)
    s = s.replace("&", " AND ")
    # remaining punctuation -> space (keeps digits & letters, drops , . - / ( ) etc)
    s = _PUNCT_TO_SPACE.sub(" ", s)
    # collapse multi-space + trim
    s = _MULTISPACE.sub(" ", s).strip()
    return s

In [6]:
# Step 1.3 : Apply cleaning
df["clean_name"] = df["original_name"].map(clean_name)
df[["original_name", "clean_name"]].head(10)

,original_name,clean_name
0,EMINENT RESEARCH & ADVISORY SERVICE,EMINENT RESEARCH AND ADVISORY SERVICE
1,S R B C & COMPANY LLP,S R B C AND COMPANY LLP
2,NAKODA NUTRIMENTS & BIOSCI PRIVATE LIMITED,NAKODA NUTRIMENTS AND BIOSCI PRIVATE LIMITED
3,WIPRO PARI ENGINEERING & SERVICES,WIPRO PARI ENGINEERING AND SERVICES
4,ESQUIRE HEALTHCARE & LGTS PRIVATE LIMITED,ESQUIRE HEALTHCARE AND LGTS PRIVATE LIMITED
5,TASTE & TRENDS HOSPITALITY PRIVATE LIMITED,TASTE AND TRENDS HOSPITALITY PRIVATE LIMITED
6,JAY MALHAR TOURS & TRAVELS,JAY MALHAR TOURS AND TRAVELS
7,KRISHNA TOURS & TRAVELS,KRISHNA TOURS AND TRAVELS
8,OM TOURS & TRAVELS,OM TOURS AND TRAVELS
9,PAVAN TOURS & TRAVELS,PAVAN TOURS AND TRAVELS


In [7]:
# Step 1.4 : Validation - inspect 100 random records; confirm no meaningful words dropped
sample = df.sample(100, random_state=42)[["original_name", "clean_name"]]
for o, c in sample.itertuples(index=False):
    print(f"{o!r:60} -> {c!r}")

'SHG WASTE MANAGEMENT PRIVATE LIMITED'                       -> 'SHG WASTE MANAGEMENT PRIVATE LIMITED'
'MANAGING DIRECTOR MOSES YESUDAS'                            -> 'MANAGING DIRECTOR MOSES YESUDAS'
'NTM SHAWLS PRIVATE LIMITED'                                 -> 'NTM SHAWLS PRIVATE LIMITED'
'VIKSON VENTURES'                                            -> 'VIKSON VENTURES'
'SDSP ENGINEERING PROJECTS LLP'                              -> 'SDSP ENGINEERING PROJECTS LLP'
'R S K CONSTRUCTION AND PROJECTS'                            -> 'R S K CONSTRUCTION AND PROJECTS'
'KAAIROSH INNOVATION OPC PRIVATE LIMITED'                    -> 'KAAIROSH INNOVATION OPC PRIVATE LIMITED'
'MEDICAL SOCIETY OF SSJA INDIA'                              -> 'MEDICAL SOCIETY OF SSJA INDIA'
'L I C OF INDIA U C SANJEEV KUMAR'                           -> 'L I C OF INDIA U C SANJEEV KUMAR'
'SHRIYUKT AGRO FARMS'                                        -> 'SHRIYUKT AGRO FARMS'
'HEELMOOI PROJECTS PRIVATE LIMITED'        

## Phase 2 : Typo / OCR Normalization
Fix common business-word OCR errors at the **token level**.
Do NOT normalize person / company / city names.

In [30]:
# Step 2.1 : OCR correction dictionary (business words only)
# Keys are wrong tokens, values are correct tokens. Extend as new OCR errors are found.
OCR_DICT = {
    "TRAVLES": "TRAVELS",
    "TRAVELES": "TRAVELS",
    "LIMITD": "LIMITED",
    "LIMTED": "LIMITED",
    "LIMITEDD": "LIMITED",
    "PRIVATELIMITED": "PRIVATE LIMITED",
    "COMPNY": "COMPANY",
    "COMPANYY": "COMPANY",
    "COOPERATIV": "COOPERATIVE",
    "INFRASTRUOTURE": "INFRASTRUCTURE",
    "INFRASTRUCTUR": "INFRASTRUCTURE",
    "ENTERPRISS": "ENTERPRISES",
    "ENTERPRISESS": "ENTERPRISES",
    "CONSTRUCTON": "CONSTRUCTION",
    "CONSTUCTION": "CONSTRUCTION",
    "INDUSTRIS": "INDUSTRIES",
    "SERVICS": "SERVICES",
    "PVT": "PRIVATE",
    "LTD": "LIMITED",
    "LIMITE": "LIMITED",
    "LIMIED": "LIMITED",
    "LMITED": "LIMITED",
    "LIMIITED": "LIMITED",
    "PRIVAT": "PRIVATE",
    "PRAVATE": "PRIVATE",
    "PRIVATED": "PRIVATE",
    "PRIVTE": "PRIVATE",
    "PRVATE": "PRIVATE",
    "PRIVETE": "PRIVATE",
    "PRIVITE": "PRIVATE",
    "TRAVELLS": "TRAVELS",
    "TRAVALS": "TRAVELS",
    "TRVELS": "TRAVELS",
    "TRAVLS": "TRAVELS",
    "TRAVELSS": "TRAVELS",
    "TRAVALES": "TRAVELS",
    "TRAVES": "TRAVELS",
    "CONTRUCTION": "CONSTRUCTION",
    "CONSTRACTION": "CONSTRUCTION",
    "CONSTRUCATION": "CONSTRUCTION",
    "CONSTRUTION": "CONSTRUCTION",
    "INFRASTUCTURE": "INFRASTRUCTURE",
    "INFRATRUCTURE": "INFRASTRUCTURE",
    "ENGINERING": "ENGINEERING",
    "ENGINNERING": "ENGINEERING",
    "ENGINEERINGS": "ENGINEERING",
    "TECHNOLGIES": "TECHNOLOGIES",
    "TECNOLOGIES": "TECHNOLOGIES",
    "TECHOLOGIES": "TECHNOLOGIES",
    "TECHNOLOGIE": "TECHNOLOGIES",
    "INDUSTIRES": "INDUSTRIES",
    "INDUTRIES": "INDUSTRIES",
    "INDUSRIES": "INDUSTRIES",
    "INDUSTRIE": "INDUSTRIES",
    "SEVICES": "SERVICES",
    "SERVIES": "SERVICES",
    "SERVISES": "SERVICES",
    "SERVIC": "SERVICES",
    "SOLUTONS": "SOLUTIONS",
    "SOLUTIO": "SOLUTIONS",
    "COMAPANY": "COMPANY",
    "COMPAN": "COMPANY",
    "TREDING": "TRADING",
    "TRANDING": "TRADING",
    "TRADINGS": "TRADING",
    "TREDERS": "TRADERS",
    "TRASPORT": "TRANSPORT",
    "HELTHCARE": "HEALTHCARE",
    "FEBRICATION": "FABRICATION",
    "JWELLERS": "JEWELLERS",
    "MACHINARY": "MACHINERY",
    "DISTRIBUTERS": "DISTRIBUTORS",
    "DEVLOPERS": "DEVELOPERS",
    "INDOTECH": "INFOTECH",
    "ENTERPRISERS": "ENTERPRISES",
    "AUTOMATIONS": "AUTOMATION",
    "MARKETINGS": "MARKETING",
    "PACKAGINGS": "PACKAGING",
    "HARDWARES": "HARDWARE",
    "AUTOMOTIVES": "AUTOMOTIVE",
    "MANUFACTURES": "MANUFACTURERS",
    "INTERNATIONALS": "INTERNATIONAL",
    "LIMTIED": "LIMITED",
    "LIITED": "LIMITED",
    "LIMIETD": "LIMITED",
    "LIMITES": "LIMITED",
    "LTMITED": "LIMITED",
    "LINITED": "LIMITED",
    "LITIMED": "LIMITED",
    "IMITED": "LIMITED",
    "LIMATED": "LIMITED",
    "LIIMTED": "LIMITED",
    "LITED": "LIMITED",
    "LIMITID": "LIMITED",
    "LEMITED": "LIMITED",
    "LOMITED": "LIMITED",
    "TIMITED": "LIMITED",
    "LIMUTED": "LIMITED",
    "LITIED": "LIMITED",
    "LEMITE": "LIMITED",
    "TLD": "LIMITED",
    "LD": "LIMITED",
    "LDT": "LIMITED",
    "LID": "LIMITED",
    "LTF": "LIMITED",
    "LTS": "LIMITED",
    "LTDF": "LIMITED",
    "LTDQ": "LIMITED",
    "LTDGRD": "LIMITED",
    "LITD": "LIMITED",
    "TLTD": "LIMITED",
    "LLTD": "LIMITED",
    "LTT": "LIMITED",
    "LTE": "LIMITED",
    "LYD": "LIMITED",
    "LTA": "LIMITED",
    "LDH": "LIMITED",
    "LLT": "LIMITED",
    "PTD": "LIMITED",
}

# Protect these from being treated as typos even if fuzzy-close (business anchors, keep as-is)
PROTECTED_TOKENS = set()

In [31]:
# Step 2.2 : Token-level replacement
def normalize_typos(clean: str) -> str:
    toks = clean.split()
    out = []
    for t in toks:
        out.append(OCR_DICT.get(t, t))   # exact-token replacement only -> safe for names/cities
    return " ".join(out)

df["typo_fixed"] = df["clean_name"].map(normalize_typos)

In [10]:
# Step 2.3 : Validation - show only rows the dictionary actually changed (up to 200)
changed = df[df["clean_name"] != df["typo_fixed"]][["clean_name", "typo_fixed"]].head(200)
print("changed rows:", (df["clean_name"] != df["typo_fixed"]).sum())
for a, b in changed.itertuples(index=False):
    print(f"{a!r:60} -> {b!r}")

changed rows: 3161
'ADITYA TOURS AND TRAVLES'                                   -> 'ADITYA TOURS AND TRAVELS'
'MAGAR S NATURAL FOOD PRIVATELIMITED'                        -> 'MAGAR S NATURAL FOOD PRIVATE LIMITED'
'KISHORE FABRICS PRIVATE LIMTED'                             -> 'KISHORE FABRICS PRIVATE LIMITED'
'SANDHYA AQUA EXPORTS PRIVATELIMITED'                        -> 'SANDHYA AQUA EXPORTS PRIVATE LIMITED'
'INDIA METALS MANUFACTURES AND TRADERS'                      -> 'INDIA METALS MANUFACTURERS AND TRADERS'
'AGRASEN TREDERS'                                            -> 'AGRASEN TRADERS'
'SYNCHEM DISTRIBUTORS PRIVATE LIMITE'                        -> 'SYNCHEM DISTRIBUTORS PRIVATE LIMITED'
'NOVITECH HEALTH CARE PRIVATELIMITED'                        -> 'NOVITECH HEALTH CARE PRIVATE LIMITED'
'STERLING INDOTECH CONSULTANTS PRIVATE L'                    -> 'STERLING INFOTECH CONSULTANTS PRIVATE L'
'S S GRAPHICS N ENTERPRISERS'                                -> 'S S GRAPHICS N ENTERPR

## Phase 3 : Canonical Name Generation
Remove legal suffixes (PRIVATE, LIMITED, PVT, LTD, LLP, AND, COMPANY, CO ...).
**Keep** industry descriptors (INFRASTRUCTURE, TRAVELS, CONSTRUCTIONS ...).

Example: `ABC TOURS AND TRAVELS PRIVATE LIMITED` -> `ABC TOURS TRAVELS`

In [ ]:
# Step 3.1 : Legal-form stopwords (stripped) vs industry descriptors (KEPT)
#
# FIX: these two sets used to be one list. Industry descriptors were being stripped from
# canonical_name, so "A J IMPEX" / "A J BUILDERS" / "A J TOURS AND TRAVELS" all collapsed to
# "A J" -> identical strings -> cosine 1.0 -> one giant false cluster. Descriptors are the
# only thing distinguishing those owners, so canonicalize() must keep them.

# Removed from canonical_name: legal form, honorifics, role words, pure connectors.
# These carry no identity at all.
LEGAL_STOPWORDS = {
    "PRIVATE", "LIMITED", "PRIVATELIMITED", "LTD", "PVT", "PVTLTD", "LLP", "LLC",
    "INC", "PLC", "CORP", "CORPORATION", "INCORPORATED", "COMPANY", "CO", "OPC",
    "HUF", "LT", "PROPRIETOR", "PROPRIETORSHIP", "PROPRIETRIX", "PROP", "PARTNER",
    "PARTNERS", "MANAGING", "DIRECTOR", "MR", "MRS", "MS", "M/S", "AND", "&",
    "THE", "OF", "FOR",
}

# KEPT in canonical_name. Weak identity signal on their own, so Phase 7 discounts them
# when computing core_tokens / core_sim -- but they still separate two owners who share
# only initials ("A J IMPEX" != "A J BUILDERS").
DESCRIPTOR_WORDS = {
    "ENTERPRISES", "ENTERPRISE", "INDUSTRIES", "INDUSTRY", "INDUSTRIAL",
    "ASSOCIATES", "AGENCIES", "AGENCY", "SERVICES", "SERVICE", "SOLUTIONS",
    "SOLUTION", "TRADERS", "TRADER", "TRADING", "TRADE", "WORKS", "WORK",
    "PRODUCTS", "GROUP", "VENTURES", "SONS", "BROTHERS", "UDYOG", "IMPEX",
    "DISTRIBUTORS", "SUPPLIERS", "DEALERS", "SALES", "MARKETING", "STORE",
    "STORES", "MART", "CENTRE", "CENTER", "HOUSE", "BUSINESS", "GENERAL",
    "MANAGEMENT", "CONSULTANTS", "CONSULTANCY", "CONSULTING", "CREATIONS",
    "INDIA", "INDIAN", "BHARAT", "INTERNATIONAL", "GLOBAL", "OVERSEAS", "WORLD",
    "NEW", "TRAVELS", "TRAVEL", "TOURS", "TOUR", "LOGISTICS", "LOGISTIC",
    "TRANSPORT", "TRANSPORTS", "ROADLINES", "MOVERS", "CARGO", "EXPRESS",
    "CONSTRUCTION", "CONSTRUCTIONS", "ENGINEERING", "ENGINEERS", "ENGG",
    "INFRA", "INFRASTRUCTURE", "INFRATECH", "BUILDCON", "BUILDERS", "BUILDING",
    "DEVELOPERS", "REALTY", "PROPERTIES", "PROJECT", "PROJECTS",
    "TECHNOLOGIES", "TECHNOLOGY", "TECH", "INFOTECH", "SYSTEMS", "SYSTEM",
    "AUTOMATION", "NETWORK", "NETWORKS", "COMMUNICATION", "COMMUNICATIONS",
    "MOTORS", "AUTO", "AUTOMOBILE", "AUTOMOBILES", "AUTOMOTIVE",
    "STEEL", "STEELS", "METAL", "METALS", "MINERALS", "SPONGE",
    "FOODS", "FOOD", "FARMS", "FARM", "AGRO", "RICE", "MILLS", "MILL", "COFFEE",
    "JEWELLERS", "JEWELLER", "JEWELLERY", "POWER", "ENERGY", "OIL", "GAS",
    "PHARMA", "PHARMACEUTICALS", "TEXTILES", "TEXTILE", "GARMENTS", "FASHION",
    "MEDICAL", "HEALTHCARE", "HEALTH", "CARE", "HOSPITAL", "HOTEL", "HOTELS",
    "HOSPITALITY", "RESORTS", "CHEMICALS", "POLYMERS", "ELECTRONICS",
    "ELECTRICALS", "ELECTRICAL", "ELECTRIC", "HARDWARE", "TOOLS", "EQUIPMENTS",
    "EQUIPMENT", "CRANE", "MARINE", "PACKAGING", "MEDIA", "FINANCE", "FIN",
    "CAPITAL", "INVESTMENT", "INVESTMENTS", "BANK", "SECURITY", "RESEARCH",
    "SCIENCE", "SCIENCES", "LIFESCIENCES", "LIFE",
    "GLASS", "STONE", "EXPORTS", "EXPORT", "IMPORT",
}

assert not (LEGAL_STOPWORDS & DESCRIPTOR_WORDS), "a word must be legal-form OR descriptor, not both"
print("legal stopwords:", len(LEGAL_STOPWORDS), "| descriptors kept:", len(DESCRIPTOR_WORDS))

In [33]:
# Step 3.2 : Canonicalization
def canonicalize(name: str) -> str:
    toks = [t for t in name.split() if t not in LEGAL_STOPWORDS]
    canon = " ".join(toks).strip()
    # guard: never return empty -> fall back to the un-stripped name
    return canon if canon else name

df["canonical_name"] = df["typo_fixed"].map(canonicalize)

# Final schema: original_name, clean_name, typo_fixed, canonical_name
df[["original_name", "clean_name", "canonical_name"]].head(10)

,original_name,clean_name,canonical_name
0,EMINENT RESEARCH & ADVISORY SERVICE,EMINENT RESEARCH AND ADVISORY SERVICE,EMINENT ADVISORY
1,S R B C & COMPANY LLP,S R B C AND COMPANY LLP,S R B C
2,NAKODA NUTRIMENTS & BIOSCI PRIVATE LIMITED,NAKODA NUTRIMENTS AND BIOSCI PRIVATE LIMITED,NAKODA NUTRIMENTS BIOSCI
3,WIPRO PARI ENGINEERING & SERVICES,WIPRO PARI ENGINEERING AND SERVICES,WIPRO PARI
4,ESQUIRE HEALTHCARE & LGTS PRIVATE LIMITED,ESQUIRE HEALTHCARE AND LGTS PRIVATE LIMITED,ESQUIRE LGTS
5,TASTE & TRENDS HOSPITALITY PRIVATE LIMITED,TASTE AND TRENDS HOSPITALITY PRIVATE LIMITED,TASTE TRENDS
6,JAY MALHAR TOURS & TRAVELS,JAY MALHAR TOURS AND TRAVELS,JAY MALHAR
7,KRISHNA TOURS & TRAVELS,KRISHNA TOURS AND TRAVELS,KRISHNA
8,OM TOURS & TRAVELS,OM TOURS AND TRAVELS,OM
9,PAVAN TOURS & TRAVELS,PAVAN TOURS AND TRAVELS,PAVAN


In [13]:
# Step 3.3 : Validation - original -> canonical for 200 random rows
for o, c in df.sample(200, random_state=7)[["original_name", "canonical_name"]].itertuples(index=False):
    print(f"{o!r:60} -> {c!r}")

'ADEELITE DESIGN CONSULTANTS PRIVATE LIMITED'                -> 'ADEELITE DESIGN'
'RENARD SERVICES PRIVATE LIMITED'                            -> 'RENARD'
'FABSTRACT CLOTHING INDIA PRIVATE LIMITED'                   -> 'FABSTRACT CLOTHING'
'GLOBUS PACKAGING'                                           -> 'GLOBUS'
'K S T AGENCY'                                               -> 'K S T'
'CARE & CURE MEDICITY HOSPITAL'                              -> 'CURE MEDICITY'
'MAA NAVDURGA CONSTRUCTION COM'                              -> 'MAA NAVDURGA COM'
'SINGH CARPET INTERNATIONAL'                                 -> 'SINGH CARPET'
'THE DESIGNATED PARTNER ELDO RAJAN'                          -> 'DESIGNATED ELDO RAJAN'
'K B METAL WORKS'                                            -> 'K B'
'MARU AUTOMOBILES'                                           -> 'MARU'
'SHREE BRAMHADEV TOURS AND TRAVELS'                          -> 'SHREE BRAMHADEV'
'SION HEALTHCARE SHL LIMITED'                                -

In [14]:
# Step 3.4 : Persist cleaned frame + drop rows with empty canonical
df = df[df["canonical_name"].str.len() > 0].reset_index(drop=True)
df["row_id"] = np.arange(len(df))          # stable id used everywhere downstream
df.to_parquet(art("names_clean.parquet"), index=False)
print("kept rows:", len(df))

kept rows: 378716


## Phase 4 : Embedding Generation
Encode `canonical_name` with Qwen embedding model, L2-normalized.
Store `embeddings.npy`. Expected shape ~ `(349413, 4096)`.

In [35]:
# Step 4.1 : Load Qwen embedding model (GPU only)
import torch
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "CUDA GPU not available - this pipeline requires a GPU"
model = SentenceTransformer(CONFIG["model_name"], device="cuda")   # force GPU; downloads on first run
print("device:", model.device, "| embedding dim:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

device: cuda:0 | embedding dim: 4096


/var/tmp/ipykernel_5670/3417500702.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("device:", model.device, "| embedding dim:", model.get_sentence_embedding_dimension())


In [16]:
# Step 4.2 : Encode canonical names (batched, normalized, on GPU)
texts = df["canonical_name"].tolist()

embeddings = model.encode(
    texts,
    batch_size=CONFIG["batch_size"],
    normalize_embeddings=CONFIG["normalize"],   # -> L2 norm == 1, so inner product == cosine
    show_progress_bar=True,
    convert_to_numpy=True,
    device="cuda",                              # force GPU encode
).astype("float32")

np.save(art("embeddings.npy"), embeddings)
print("shape:", embeddings.shape)

Batches:   0%|          | 0/1480 [00:00<?, ?it/s]

shape: (378716, 4096)


In [17]:
# Step 4.3 : Validation - shape + L2 norm ~= 1
embeddings = np.load(art("embeddings.npy"))
norms = np.linalg.norm(embeddings, axis=1)
print("shape:", embeddings.shape)
print("norm min/max:", norms.min(), norms.max())   # expect ~1.0 both
assert embeddings.shape[0] == len(df)

shape: (378716, 4096)
norm min/max: 0.9962626 1.0039062


## Phase 5 : FAISS Index
Build `IndexFlatIP` (exact inner-product = cosine on normalized vectors). Search top-100 neighbors.

In [37]:
# Step 5.1 : Build index
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)     # exact search, cosine via normalized IP
index.add(embeddings)
print("indexed vectors:", index.ntotal)

faiss.write_index(index, art("faiss_flatip.index"))

NameError: name 'embeddings' is not defined

In [19]:
# Step 5.2 : Search top-K neighbors for every name
# scores[i], neighbors[i] -> the K most similar rows to row i (includes itself at rank 0)
K = CONFIG["top_k"]
scores, neighbors = index.search(embeddings, K)   # both shape (N, K)

np.save(art("faiss_scores.npy"), scores)
np.save(art("faiss_neighbors.npy"), neighbors)
print("scores:", scores.shape, "neighbors:", neighbors.shape)

scores: (378716, 100) neighbors: (378716, 100)


In [20]:
# Step 5.3 : Validation - self-similarity ~= 1 for 50 random rows
for i in np.random.RandomState(0).randint(0, len(df), 50):
    # self should be rank-0 neighbor with score ~1.0
    self_pos = np.where(neighbors[i] == i)[0]
    s = scores[i][self_pos[0]] if len(self_pos) else float("nan")
    print(f"row {i:>7}  self_sim={s:.4f}  {df.canonical_name.iloc[i]!r}")

row  305711  self_sim=1.0054  'ARKISH JEWELS'
row  117952  self_sim=1.0070  'SURJIT GOODS CARRIERS'
row  152315  self_sim=1.0009  'VIDHYA'
row  358083  self_sim=1.0066  'INDUSMEDICA'
row  359783  self_sim=1.0075  'SHRAY'
row  304137  self_sim=0.9994  'SHAMBHAVI'
row  122579  self_sim=0.9996  'SAIVI'
row   86293  self_sim=0.9963  'MAXIMA'
row  374564  self_sim=1.0058  'ANUSANDHAN PLASTICS'
row  211543  self_sim=1.0062  'MEGTHINK'
row  212038  self_sim=1.0055  'BANERJEE'
row  310744  self_sim=1.0054  'ALIEN'
row  170584  self_sim=0.9978  'ALGEN'
row  314764  self_sim=1.0024  'MPSELECTROPLAST'
row   80186  self_sim=0.9994  'ELENTEC'
row   17089  self_sim=0.9990  'INTELLIGENCEPLUS EDUVISION'
row  150055  self_sim=1.0056  'JAMUNA HYGIENE CONCEPTS'
row  370775  self_sim=1.0053  'K MARK'
row  220760  self_sim=0.9982  'HINDUSTAN LABORATORIES'
row  363345  self_sim=1.0010  'WEAR'
row  255653  self_sim=1.0045  'ACCUTECH'
row   82457  self_sim=0.9975  'AZIMUTH MARITIME'
row  329843  self_sim=1.00

## Phase 6 : Candidate Pair Generation
Turn neighbor lists into unique undirected candidate pairs.
Drop self-matches and duplicate `(i,j)`/`(j,i)` edges. Apply a coarse cosine gate to cut noise.

In [21]:
# Step 6.1 : Build unique (i, j, cosine) candidate pairs
gate = CONFIG["rules"]["min_cosine_gate"]
seen = set()
pair_i, pair_j, pair_cos = [], [], []

for i in tqdm(range(neighbors.shape[0]), desc="pairs"):
    for rank in range(neighbors.shape[1]):
        j = int(neighbors[i, rank])
        c = float(scores[i, rank])
        if j == i:            # skip self-match
            continue
        if c < gate:          # coarse cosine gate -> prune obvious non-matches early
            continue
        a, b = (i, j) if i < j else (j, i)   # canonical ordering -> dedup (i,j)==(j,i)
        if (a, b) in seen:
            continue
        seen.add((a, b))
        pair_i.append(a); pair_j.append(b); pair_cos.append(c)

pairs = pd.DataFrame({"i": pair_i, "j": pair_j, "cosine": pair_cos})
print("candidate pairs:", len(pairs))

pairs:   0%|          | 0/378716 [00:00<?, ?it/s]

candidate pairs: 10206471


In [22]:
# Step 6.2 : Validation - no self, no duplicate undirected edge
assert (pairs["i"] != pairs["j"]).all(), "self-match leaked"
assert not pairs.duplicated(subset=["i", "j"]).any(), "duplicate edge leaked"
assert (pairs["i"] < pairs["j"]).all(), "ordering broken"
print("candidate pairs OK:", len(pairs))

candidate pairs OK: 10206471


## Phase 7 : Feature Engineering
For every candidate pair compute string + token + semantic features.

Plan's 9 features: cosine, rapidfuzz, token_sort, token_set, levenshtein, jaccard, common_tokens, prefix_match, biz_kw_match.
Plus 3 anti-over-merge features the Rule Engine needs: **`core_shared`** (# shared discriminative tokens),
**`core_sim`** (similarity of the name with generic words stripped, initials kept), **`single_token`** (person/brand single-word flag).

In [ ]:
# Step 7.1 : Generic-word set used by the features, and token helpers
#
# BUSINESS_KEYWORDS = legal forms + industry descriptors. These words DO NOT identify a
# company on their own, so a pair overlapping ONLY on them is not evidence of a same owner.
# Descriptors stay in canonical_name (Step 3.1) but are discounted here.
BUSINESS_KEYWORDS = LEGAL_STOPWORDS | DESCRIPTOR_WORDS

def _fold(tok):
    # light singular/plural fold so ENTERPRISE == ENTERPRISES when comparing tokens
    return tok[:-1] if len(tok) > 4 and tok.endswith("S") else tok

def core_tokens(name):
    # discriminative tokens only: drop generic words AND single-char initials
    return {_fold(t) for t in name.split() if t not in BUSINESS_KEYWORDS and len(t) > 1}

def strip_business(name):
    # remove generic words but KEEP initials -> used to compare the distinguishing part
    return " ".join(t for t in name.split() if t not in BUSINESS_KEYWORDS)

def is_single_token(name):
    return len(name.split()) == 1

def descriptor_tokens(name):
    # the industry-line words only (TOURS, STEEL, MOTORS, ...); legal forms excluded
    return {t for t in name.split() if t in DESCRIPTOR_WORDS}


def _descriptors_compatible(x, y):
    """Two descriptor words describe the same business line?

    Identical, one an abbreviation/prefix of the other (FIN/FINANCE, INFRA/INFRASTRUCTURE,
    TRAVEL/TRAVELS), or plainly similar (ENGINEERING/ENGINEERS). STEEL/MOTORS is neither.
    """
    if x == y:
        return True
    short, long_ = (x, y) if len(x) <= len(y) else (y, x)
    if len(short) >= 3 and long_.startswith(short):
        return True
    return fuzz.ratio(x, y) >= 80


def descriptors_conflict(a, b):
    """True when both names carry descriptors and NONE of them line up.

    "TATA STEEL" vs "TATA MOTORS" -> same brand, different business -> different owners.
    Returns False if either side has no descriptor at all (nothing to compare).
    """
    da, db = descriptor_tokens(a), descriptor_tokens(b)
    if not da or not db:
        return False
    return not any(_descriptors_compatible(x, y) for x in da for y in db)

print("business keywords:", len(BUSINESS_KEYWORDS))
# sanity: descriptors now survive canonicalization, so these are distinguishable
for n in ["A J IMPEX", "A J BUILDERS", "A J TOURS TRAVELS"]:
    print(f"  {n!r:22} core={core_tokens(n)}  desc={descriptor_tokens(n)}")
print("  TATA STEEL vs TATA MOTORS  conflict:", descriptors_conflict("TATA STEEL", "TATA MOTORS"))
print("  CHOLA INV FIN vs CHOLA INVEST FINANCE conflict:",
      descriptors_conflict("CHOLAMANDALAM INV FIN", "CHOLAMANDALAM INVEST FINANCE"))

In [ ]:
# Step 7.2 : Per-pair feature computation
def pair_features(a: str, b: str, cosine: float) -> dict:
    ta, tb = set(a.split()), set(b.split())
    inter, union = ta & tb, ta | tb

    ca, cb = core_tokens(a), core_tokens(b)     # discriminative tokens (no generic words, no initials)
    core_inter = ca & cb

    # similarity of the DISTINGUISHING part (generic words removed, initials kept).
    # "G S R" vs "J S" -> low ; "NOKIA SOLUTIONS ..." vs "NOKIA SOLUTION ..." -> high.
    sa, sb = strip_business(a), strip_business(b)
    core_sim = max(
        fuzz.token_set_ratio(sa, sb),
        fuzz.ratio(sa.replace(" ", ""), sb.replace(" ", "")),  # catches concatenation (NOKIASOLUTIONS)
    )

    return {
        "cosine":        cosine,
        "rapidfuzz":     fuzz.ratio(a, b),                 # raw char ratio
        "token_sort":    fuzz.token_sort_ratio(a, b),      # order-insensitive
        "token_set":     fuzz.token_set_ratio(a, b),       # set-based
        "levenshtein":   Levenshtein.distance(a, b),       # edit distance
        "jaccard":       (len(inter) / len(union)) if union else 0.0,
        "common_tokens": len(inter),
        "prefix_match":  int(a[:3] == b[:3]),              # same first 3 chars
        "biz_kw_match":  int(len(inter) > 0 and len(core_inter) == 0),  # overlap is only generic words
        "core_shared":   len(core_inter),                  # # shared discriminative tokens
        "core_sim":      core_sim,                         # similarity of the distinguishing part
        "single_token":  int(is_single_token(a) or is_single_token(b)),  # person/brand single word
        # NEW: either side has NO discriminative token at all -> "A J IMPEX" is just
        # initials + a generic word. Nothing here can prove two such owners are the same,
        # so Step 8.1 demands character-identical canonicals for these pairs.
        "core_empty":    int(len(ca) == 0 or len(cb) == 0),
        # NEW: same brand, unrelated business line -> TATA STEEL vs TATA MOTORS.
        # strip_business() reduces both to "TATA", so core_sim alone cannot see this.
        "desc_conflict": int(descriptors_conflict(a, b)),
    }

In [25]:
# Step 7.3 : Build feature table (vectorized over pairs)
canon = df["canonical_name"].to_numpy()

feat_rows = []
for i, j, c in tqdm(pairs.itertuples(index=False), total=len(pairs), desc="features"):
    feat_rows.append(pair_features(canon[i], canon[j], c))

features = pd.DataFrame(feat_rows)
features.insert(0, "i", pairs["i"].values)
features.insert(1, "j", pairs["j"].values)
features.insert(2, "owner1", canon[pairs["i"].values])
features.insert(3, "owner2", canon[pairs["j"].values])

features.to_parquet(art("features.parquet"), index=False)
features.head(20)

features:   0%|          | 0/10206471 [00:00<?, ?it/s]

,i,j,owner1,owner2,cosine,rapidfuzz,token_sort,token_set,levenshtein,jaccard,common_tokens,prefix_match,biz_kw_match,core_shared,core_sim,single_token
0,0,369310,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
1,0,364305,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
2,0,342914,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
3,0,301785,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
4,0,299496,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
5,0,267715,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
6,0,249941,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
7,0,227361,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
8,0,220191,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1
9,0,150372,EMINENT ADVISORY,EMINENT,0.850956,60.869565,60.869565,100.0,9,0.50,1,1,0,1,100.000000,1


In [26]:
# Step 7.4 : Validation - inspect 500 random pairs manually
features.sample(500, random_state=1)[
    ["owner1", "owner2", "cosine", "token_set", "rapidfuzz", "levenshtein",
     "core_sim", "core_shared", "single_token", "biz_kw_match"]
]

,owner1,owner2,cosine,token_set,rapidfuzz,levenshtein,core_sim,core_shared,single_token,biz_kw_match
10192295,AAKASH AGARWAL,SIDDHARTH AGARWAL,0.873579,66.666667,64.516129,7,66.666667,1,0,0
7138542,S R M,S R,0.922389,100.000000,75.000000,2,100.000000,0,0,1
5576193,SAI PRASAD,SAI RAMESHWAR,0.905311,69.565217,69.565217,6,69.565217,1,0,0
2200452,MSK,MSKP,0.882954,85.714286,85.714286,1,85.714286,0,1,0
6306557,V R,V R,1.007111,100.000000,100.000000,0,100.000000,0,0,1
...,...,...,...,...,...,...,...,...,...,...
1385685,SIDHESHWAR,SHREE SIDDHESHWAR,0.865253,74.074074,74.074074,7,76.923077,0,1,0
152151,SOMESH,SOMENDRA,0.884100,57.142857,57.142857,4,57.142857,0,1,0
6658377,AGRAWAL,AGRAWAL,1.007392,100.000000,100.000000,0,100.000000,1,1,0
7308016,JAI DURGA,JAI DURGE,0.882540,88.888889,88.888889,1,88.888889,1,0,0


## Phase 8 : Rule Engine
Never merge on cosine alone. **Guards run before any merge path** — they kill the over-merge
patterns short embeddings cause (Qwen rates `G S R CONSTRUCTIONS` and `J S CONSTRUCTIONS` ~0.98
because the generic word dominates the vector).

**Guards (no similarity score overrides these)**
- **(a) single-token guard**: if either name is one token (person/brand: `RAJ`, `KRISHNA`, `SHIVAM`) merge ONLY if identical (`lev == 0`). Kills `RAJ`/`RAJU`, `KRISHNA`/`KRISH`.
- **(b) core-empty guard**: if either name has **no** discriminative token (initials + a generic word, e.g. `A J IMPEX`), merge ONLY if the canonicals are character-identical. This is what stops `A J IMPEX` / `A J BUILDERS` / `A J TOURS TRAVELS` from fusing into one owner.
- **(c) descriptor-conflict guard**: shared brand but disjoint business lines -> reject. `TATA STEEL` vs `TATA MOTORS`, `RELIANCE INDUSTRIES` vs `RELIANCE POWER`. Abbreviations and plurals still count as the same line (`FIN`/`FINANCE`, `TRAVEL`/`TRAVELS`, `ENGINEERING`/`ENGINEERS`).

**Merge path that runs next**
- **Near-exact**: rapidfuzz >= 95 AND cosine >= 0.97 -> same name, different spacing/concat (`NOKIASOLUTIONS NETWORKSINDIA`). Sits below the guards, above the remaining rejects.

**Hard rejects**
- **(d) distinguishing-part guard**: generic words stripped, initials kept -> `core_sim` must be >= 85. `G S R` vs `J S` -> low -> reject.
- **(e) no shared meaningful token**: `core_shared == 0` -> reject. `RAJA STEELS` vs `RAHUL STEELS`.

**Merge rules**
- **Rule 1**: cosine >= 0.97 AND token_set >= 95 AND rapidfuzz >= 90
- **Rule 2**: levenshtein <= 2 AND cosine >= 0.94
- **Rule 3**: cosine >= 0.95 AND core_sim >= 90 (abbreviation expansion, e.g. `INV FIN` -> `INVEST FINANCE`)

In [ ]:
# Step 8.0 : RESUME from saved files (run in a FRESH kernel, after Step 0)
# Phase 8 onward uses these saved files, not upstream cell variables.
# After this cell, run Step 8.1 (defines `decide`), then 8.2 onward.
#   (Step 8.1 needs only CONFIG - already set in Step 0.)
#
# WARNING: features.parquet must have been built by the CURRENT Step 3.1 / 7.1 / 7.2 code.
# The stopword split (descriptors kept) and the new core_empty / desc_conflict features both
# changed the schema, so an older artifact fails the assert below - re-run Phases 1-7.
# Phase 10 (Step 10.1) additionally needs clean_name from Step 1.2.

df = pd.read_parquet(art("names_clean.parquet"))     # original_name, canonical_name, row_id
canon = df["canonical_name"].to_numpy()

features = pd.read_parquet(art("features.parquet"))   # i, j, owner1, owner2, + all pair features

assert features["i"].max() < len(df) and features["j"].max() < len(df), \
    "features reference row ids beyond df - artifacts out of sync"
_missing = {"core_empty", "desc_conflict"} - set(features.columns)
assert not _missing, f"features.parquet predates the over-merge fix (no {_missing}) - re-run Phases 1-7"
print("loaded df:", len(df), "| features:", len(features))

In [ ]:
# Step 8.1 : Decision function per pair (precision-first: reject before merge)
#
# ORDER MATTERS. The single-token / core_empty / desc_conflict guards run ABOVE the
# near-exact bypass, because a bypass on cosine+rapidfuzz alone cannot tell "A J IMPEX"
# from "A J BUILDERS" (previously both canonicalized to "A J" -> lev 0 -> 38 owners fused).
# The remaining rejects run below the bypass so genuine concatenation variants
# ("NOKIASOLUTIONS NETWORKSINDIA" vs "NOKIA SOLUTIONS NETWORKS INDIA") still merge.
R = CONFIG["rules"]

def decide(f) -> bool:
    # ---- Guards that no similarity score may override ----
    # (a) single-token person/brand names -> merge only if identical
    if f["single_token"] and f["levenshtein"] != 0:
        return False                     # RAJ vs RAJU, KRISHNA vs KRISH, SHIVAM vs SHIVANSHI

    # (b) no discriminative token on either side (initials + generic word only).
    #     Merge ONLY on a character-identical canonical: "A J IMPEX" == "A J IMPEX PVT LTD"
    #     merges, "A J IMPEX" vs "A J BUILDERS" does not.
    if f["core_empty"]:
        return f["levenshtein"] == 0

    # (c) shared brand but disjoint business lines -> different owners.
    if f["desc_conflict"]:
        return False                     # TATA STEEL vs TATA MOTORS, RELIANCE INDUSTRIES vs RELIANCE POWER

    # near-exact full string -> same name with different spacing/concatenation
    if f["rapidfuzz"] >= R["near_exact_rf"] and f["cosine"] >= R["near_exact_cos"]:
        return True                      # NOKIASOLUTIONS NETWORKSINDIA == NOKIA SOLUTIONS NETWORKS INDIA

    # ---- Hard rejects ----
    # (d) distinguishing part too different -> <initials> + generic word only
    if f["core_sim"] < R["core_sim_min"]:
        return False                     # G S R CONSTRUCTIONS vs J S CONSTRUCTIONS
    # (e) must share at least one discriminative (brand/name) token
    if f["core_shared"] == 0:
        return False                     # RAJA STEELS vs RAHUL STEELS

    # ---- Merge rules (a shared discriminative token is now guaranteed) ----
    # Rule 1
    if f["cosine"] >= R["r1_cosine"] and f["token_set"] >= R["r1_token_set"] and f["rapidfuzz"] >= R["r1_rapidfuzz"]:
        return True
    # Rule 2
    if f["levenshtein"] <= R["r2_lev"] and f["cosine"] >= R["r2_cosine"]:
        return True
    # Rule 3 : abbreviation expansion around a shared brand token (CHOLAMANDALAM INV FIN == ... INVEST FINANCE)
    if f["cosine"] >= R["r3_cosine"] and f["core_sim"] >= R["r3_core_sim"]:
        return True
    return False

In [5]:
# Step 8.2 : Apply rules -> accepted / rejected masks
features["merge"] = features.apply(decide, axis=1)
accepted = features[features["merge"]].copy()
rejected = features[~features["merge"]].copy()

print("accepted:", len(accepted), " rejected:", len(rejected))
accepted.to_parquet(art("accepted_pairs.parquet"), index=False)

accepted: 1368099  rejected: 8838372


In [6]:
# Step 8.3 : Validation - inspect 1000 accepted + 1000 rejected, tune thresholds
print("===== ACCEPTED sample =====")
display(accepted.sample(min(1000, len(accepted)), random_state=2)[
    ["owner1", "owner2", "cosine", "token_set", "rapidfuzz", "levenshtein"]])

print("===== REJECTED sample =====")
display(rejected.sample(min(1000, len(rejected)), random_state=3)[
    ["owner1", "owner2", "cosine", "token_set", "rapidfuzz", "levenshtein",
     "core_sim", "core_shared", "single_token"]])

===== ACCEPTED sample =====


,owner1,owner2,cosine,token_set,rapidfuzz,levenshtein
1808566,ATUL,ATUL,1.005700,100.000000,100.000000,0
7933569,R2,R2,1.004382,100.000000,100.000000,0
3734353,KHANNA,KHANNA,0.997660,100.000000,100.000000,0
3853527,DIGITAL,DIGITAL,1.005166,100.000000,100.000000,0
2432363,TYAGI,TYAGI,1.000697,100.000000,100.000000,0
...,...,...,...,...,...,...
7604663,SILICA,SILICA,1.002753,100.000000,100.000000,0
7403657,R R,R R,1.006426,100.000000,100.000000,0
2282289,JAN SHIKSHAN SANSTHAN,JANSHIKSHAN SANSTHAN,0.959051,97.560976,97.560976,1
348777,NOKIA NETWORK,NOKIA NETWORK,0.996353,100.000000,100.000000,0


===== REJECTED sample =====


,owner1,owner2,cosine,token_set,rapidfuzz,levenshtein,core_sim,core_shared,single_token
8062245,AARTH REALTECH,AKSHAR SHANTI REALTORS,0.901146,55.555556,55.555556,12,55.555556,0,0
6860032,YUG,YGL,0.850568,66.666667,66.666667,2,66.666667,0,1
7785054,SHREE SHYAM CREATION,SHREE SHYAM,0.933884,100.000000,70.967742,9,100.000000,2,0
9765895,S R T,SRT BLUE,0.858889,46.153846,46.153846,6,60.000000,0,0
7564707,GAJALAXMI,GAJALAKSHMI,0.906382,80.000000,80.000000,3,80.000000,0,1
...,...,...,...,...,...,...,...,...,...
5008789,RANVEET,RANA,0.885899,54.545455,54.545455,4,54.545455,0,1
10036196,K K,K K V,0.915541,100.000000,75.000000,2,100.000000,0,0
6940673,ALAYANA,ALVEERA,0.852485,42.857143,42.857143,4,42.857143,0,1
5430080,SATISH JAIN KOSWAL,SATISH,0.915436,100.000000,50.000000,12,100.000000,1,1


## Phase 9 : Union-Find (Disjoint Set)
Merge only approved pairs into clusters. Output `cluster_id` per name.

In [7]:
# Step 9.1 : Union-Find with path compression + union by rank
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]   # path compression
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1

In [8]:
# Step 9.2 : Merge approved pairs
uf = UnionFind(len(df))
for i, j in accepted[["i", "j"]].itertuples(index=False):
    uf.union(int(i), int(j))

# assign cluster_id = root of each element
df["cluster_id"] = [uf.find(x) for x in range(len(df))]
df[["canonical_name", "cluster_id"]].head(10)

,canonical_name,cluster_id
0,EMINENT ADVISORY,0
1,S R B C,1
2,NAKODA NUTRIMENTS BIOSCI,2
3,WIPRO PARI,3
4,ESQUIRE LGTS,4
5,TASTE TRENDS,5
6,JAY MALHAR,6
7,KRISHNA,7
8,OM,8
9,PAVAN,9


In [9]:
# Step 9.3 : Validation - cluster size distribution
sizes = df["cluster_id"].value_counts()
print("total clusters :", sizes.shape[0])
print("singletons     :", (sizes == 1).sum())
print("avg size       :", round(sizes.mean(), 3))
print("largest 10     :\n", sizes.head(10))

total clusters : 259579
singletons     : 227470
avg size       : 1.459
largest 10     :
 cluster_id
629      262
11637    223
675      199
7        197
890      184
4247     179
8        177
5519     174
1284     162
28       156
Name: count, dtype: int64


In [10]:
# Step 9.4 : Inspect clusters with > 20 members (over-merge check)
big = sizes[sizes > 20].index
for cid in list(big)[:20]:
    members = df.loc[df.cluster_id == cid, "canonical_name"].tolist()
    print(f"\n--- cluster {cid} ({len(members)}) ---")
    for m in members[:30]:
        print("   ", m)


--- cluster 629 (262) ---
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI
    BALAJI

--- cluster 11637 (223) ---
    SHREE SAI
    SHREE SAI
    SWAMI SAMRTH
    SREE SAI
    SREE SAI UMA SANKAR
    SHIRDI SAI
    SHREE SWAMEE SMARTH
    SHREE SWAMI SAMARTHA
    SREE SAI
    SHREE SAI
    SHREE SAI
    SWAMI SAMARTH
    SHREE SAI SAMARTH
    SREE SAI
    SAI GURU
    SREE GURU SAI
    SHIVA SAI
    SHREE SWAMI SMARTHA
    SHREE SAI
    SHREE SWAMI SAMARATH
    SWAMI SAMARTH
    SHIRDI SAI
    SWAMI SAMARATH
    SHREE SAI
    SHREE SWAMI SMARTH
    SAI SIVA
    SHIRDI SAI
    SHREE SWAMI SAMARTHA
    SREE SAI
    SHREE SAI

--- cluster 675 (199) ---
    SAI
    SAI
    SAI
    SAI
    SAI
    SAI
    SAI
    SAI
    SA

## Phase 10 : Canonical Owner Name Selection
Pick one representative owner per cluster. Default heuristic: most frequent original name, tie-break by longest.

In [ ]:
# Step 10.1 : Choose one representative owner per cluster
#
# Self-contained: safe to run in a kernel that resumed at Step 8.0 and never executed
# Phase 1. It re-defines the Step 1.2 cleaner if that name is missing, instead of
# raising NameError halfway through Phase 10.
#
# Needs only: pandas + `df` carrying original_name and cluster_id (from Step 9.2).

import os, re, unicodedata
import pandas as pd

# ---- prerequisites -------------------------------------------------------
try:
    df
except NameError:
    raise RuntimeError("`df` not in memory - run Step 8.0 (resume), then 8.1, 8.2, 9.1, 9.2")

for _c in ("original_name", "cluster_id"):
    assert _c in df.columns, f"df has no `{_c}` column - Step 9.2 must run before Phase 10"

if "art" not in globals():                      # Step 0.3 not re-run after a restart
    _ART_DIR = globals().get("CONFIG", {}).get("artifacts_dir", "artifacts")
    os.makedirs(_ART_DIR, exist_ok=True)
    def art(name):
        return os.path.join(_ART_DIR, name)

if "clean_name" not in globals():               # inline copy of Step 1.2
    _PUNCT_TO_SPACE = re.compile(r"[^A-Z0-9&\s]")
    _MULTISPACE     = re.compile(r"\s+")

    def clean_name(name: str) -> str:
        if not isinstance(name, str):
            return ""
        s = unicodedata.normalize("NFKD", name)
        s = "".join(ch for ch in s if not unicodedata.combining(ch))
        s = s.upper().replace("&", " AND ")
        s = _PUNCT_TO_SPACE.sub(" ", s)
        return _MULTISPACE.sub(" ", s).strip()

# ---- label quality -------------------------------------------------------
# Demote honorific fragments so "& MRS A J TOURS AND TRAVEL" cannot win a cluster just
# for being the longest string. Such a name is only used when nothing cleaner exists.
_HONORIFICS  = {"MR", "MRS", "MS", "M/S", "SHRI", "SHREE", "SMT", "SRI", "MISS", "DR"}
_BAD_LEADING = {"AND", "OF", "THE", "FOR"}      # "& MRS ..." -> cleans to "AND MRS ..."

def _is_clean_label(name: str) -> bool:
    toks = clean_name(name).split()
    if not toks or toks[0] in _BAD_LEADING:
        return False
    return not (_HONORIFICS & set(toks))

# ---- pick the representative --------------------------------------------
# Ranking: clean label first, then most frequent original_name, tie-break on longest.
# Vectorized on purpose - a groupby.apply with a Python picker runs one call per cluster
# and takes minutes at ~337k clusters; this is two sorts.
cand = (df.groupby(["cluster_id", "original_name"], sort=False)
          .size().rename("freq").reset_index())

_ok = {n: _is_clean_label(n) for n in cand["original_name"].unique()}
cand["clean_ok"] = cand["original_name"].map(_ok).astype(int)
cand["nlen"]     = cand["original_name"].str.len()

cand = cand.sort_values(
    ["cluster_id", "clean_ok", "freq", "nlen"],
    ascending=[True, False, False, False],
    kind="mergesort",
)

cluster_canon = (
    cand.drop_duplicates("cluster_id")[["cluster_id", "original_name"]]
        .rename(columns={"original_name": "canonical_owner"})
        .reset_index(drop=True)
)
# NOTE: map on cluster_canon["cluster_id"]. The previous version mapped over
# df["cluster_id"] - a len(df) series assigned into a len(clusters) frame, aligned
# positionally, so most cluster_size values were wrong.
cluster_canon["cluster_size"] = cluster_canon["cluster_id"].map(df["cluster_id"].value_counts())

assert cluster_canon["cluster_id"].is_unique
assert cluster_canon["cluster_size"].sum() == len(df), "cluster sizes do not cover every row"

_dirty = int((~cand.drop_duplicates("cluster_id")["clean_ok"].astype(bool)).sum())
print("clusters:", len(cluster_canon), "| clusters whose only label is an honorific fragment:", _dirty)
cluster_canon.head(10)

In [ ]:
# Step 10.2 : Attach canonical owner back to every row + persist
df = df.drop(columns=["canonical_owner"], errors="ignore")   # idempotent if this cell re-runs
df = df.merge(cluster_canon[["cluster_id", "canonical_owner"]], on="cluster_id", how="left")
assert df["canonical_owner"].notna().all(), "some rows got no canonical owner"

df.to_parquet(art("names_clustered.parquet"), index=False)
cluster_canon.to_parquet(art("cluster_canonical.parquet"), index=False)
cluster_canon.sort_values("cluster_size", ascending=False).head(20)

In [13]:
# Step 10.3 : Validation - inspect largest 100 clusters
top100 = cluster_canon.sort_values("cluster_size", ascending=False).head(100)
for cid, owner, sz in top100.itertuples(index=False):
    members = df.loc[df.cluster_id == cid, "canonical_name"].unique()[:15]
    print(f"\n[{cid}] size={sz} canonical={owner!r}")
    for m in members:
        print("   ", m)


[135985] size=262 canonical='HANDA COLD STORE'
    HANDA COLD

[269970] size=262 canonical='MUHUNTHAN MILLS'
    MUHUNTHAN

[32016] size=262 canonical='STAR FACILITIES MANAGEMENT LIMITED'
    STAR FACILITIES

[177233] size=262 canonical='SAGAR GRANITE AND MARBLE'
    SAGAR GRANITE MARBLE

[349832] size=262 canonical='A N SHAH'
    A N SHAH

[280328] size=262 canonical='PROPRIETOR FAZIL S'
    FAZIL S

[4239] size=262 canonical='SHREE PUSHKAR CHEMICALS & FERT LIMITED'
    SHREE PUSHKAR FERT

[49651] size=262 canonical='CRITICALCARE HOSPITAL PRIVATE LIMITED'
    CRITICALCARE

[304784] size=262 canonical='CALIGRA ENTERPRISE AND BUSINESS'
    CALIGRA

[244603] size=262 canonical='BHARTIYA JAN UTHAN PARISHAD'
    BHARTIYA JAN UTHAN PARISHAD

[36041] size=262 canonical='NATHKRUPA TOURS AND TRAVELS'
    NATHKRUPA

[303018] size=262 canonical='R S BUILDING MATERIALS'
    R S MATERIALS

[65447] size=262 canonical='ACTION EARTH MOVERS'
    ACTION EARTH

[347701] size=262 canonical='ESHWA SHIPPI

In [17]:
df.loc[df["cluster_id"] == 140603]

,original_name,clean_name,typo_fixed,canonical_name,row_id,cluster_id,canonical_owner


In [27]:
df.loc[df["canonical_owner"].str.contains("MICROSOFT")]["canonical_owner"].value_counts()

canonical_owner
MICROSOFT GLOBAL SERVICE CENTER INDIA PRIVATE LIMITED    16
MICROSOFT INDIA R D PRIVATE LIMITED                       8
MICROSOFT RESEARCH LAB INDIA PRIVATE LIMITED              3
MICROSOFT GLOBAL SERVICE CENTER PRIVATE L                 2
MICROSOFT G S PRIVATE LT                                  1
MICROSOFT COR INDIA                                       1
MICROSOFT INDIA R&D PRIVATE LIMITED-M APPARAO             1
MICROSOFT I R D PRIVATE LIMITED                           1
MICROSOFT CORPORATION INDIA PRIVA                         1
MICROSOFT INDIA R AN                                      1
MICROSOFT RESEARCHLAB INDIA PRIVATE LIMITED               1
MICROSOFTIND R&D PRIVATE                                  1
MICROSOFT GLOBAL SE CENTER INDIA PRIVATE LIMITED          1
MICROSOFT GSC INDIA PRIVATE LIMITED                       1
Name: count, dtype: int64

In [22]:
# import pandas as pd

df = pd.read_parquet("artifacts/names_clustered.parquet")
df.head()

,original_name,clean_name,typo_fixed,canonical_name,row_id,cluster_id,canonical_owner
0,EMINENT RESEARCH & ADVISORY SERVICE,EMINENT RESEARCH AND ADVISORY SERVICE,EMINENT RESEARCH AND ADVISORY SERVICE,EMINENT ADVISORY,0,0,EMINENT RESEARCH & ADVISORY SERVICE
1,S R B C & COMPANY LLP,S R B C AND COMPANY LLP,S R B C AND COMPANY LLP,S R B C,1,1,S R B C & ASSOCIATES LLP
2,NAKODA NUTRIMENTS & BIOSCI PRIVATE LIMITED,NAKODA NUTRIMENTS AND BIOSCI PRIVATE LIMITED,NAKODA NUTRIMENTS AND BIOSCI PRIVATE LIMITED,NAKODA NUTRIMENTS BIOSCI,2,2,NAKODA NUTRIMENTS & BIOSCI PRIVATE LIMITED
3,WIPRO PARI ENGINEERING & SERVICES,WIPRO PARI ENGINEERING AND SERVICES,WIPRO PARI ENGINEERING AND SERVICES,WIPRO PARI,3,3,WIPRO PARI ENGINEERING AND SERVICE PRIVATE LIM...
4,ESQUIRE HEALTHCARE & LGTS PRIVATE LIMITED,ESQUIRE HEALTHCARE AND LGTS PRIVATE LIMITED,ESQUIRE HEALTHCARE AND LGTS PRIVATE LIMITED,ESQUIRE LGTS,4,4,ESQUIRE HEALTHCARE & LGTS PRIVATE LIMITED


In [23]:
df["cluster_id"].nunique()

259579

In [ ]:
# df.loc[df["canonical_owner"].str.contains("BAJAJ", na=False)]

In [24]:
unq_clusters = df.drop_duplicates("cluster_id")

unq_clst_df = unq_clusters[["cluster_id","canonical_owner"]]

In [25]:
unq_clst_df.to_csv("unique_cluster_with_name_v2.csv",index=False)

## Phase 12 : Incremental Pipeline
Resolve a **new** owner against the existing index without re-embedding everything.
clean -> canonical -> embed -> search existing FAISS -> features -> rules -> join existing cluster OR create new.

In [40]:
# Step 12.0 : Load saved artifacts (run in a FRESH kernel, after Step 0)
# Incremental resolution needs the trained state back in memory. This cell reloads it from disk
# so you do NOT re-run Phases 1-11.
#
# PREREQUISITE — also execute these function-defining cells (they are just `def`s, cheap):
#   Step 1.2 clean_name | Step 2.2 normalize_typos | Step 3.2 canonicalize
#   Step 7.1 BUSINESS_KEYWORDS + core_tokens/strip_business/is_single_token
#   Step 7.2 pair_features | Step 8.1 decide
import torch
import faiss
from sentence_transformers import SentenceTransformer

# 1. embedding model (GPU only; weights cached after first download)
# assert torch.cuda.is_available(), "CUDA GPU not available - this pipeline requires a GPU"
# model = SentenceTransformer(CONFIG["model_name"], device="cuda")

# 2. FAISS index (exact same object saved in Step 5.1)
index = faiss.read_index(art("faiss_flatip.index"))
print("index vectors:", index.ntotal)

# 3. clustered names -> gives canonical_name + cluster_id per row (saved in Step 10.2)
df = pd.read_parquet(art("names_clustered.parquet"))
canon = df["canonical_name"].to_numpy()

# 4. embeddings matrix -> needed only to APPEND new vectors when persisting (Step 12.2)
embeddings = np.load(art("embeddings.npy"))

# 5. OPTIONAL batch artifacts (not needed to resolve a single new owner; load for re-analysis only)
#    scores    = np.load(art("faiss_scores.npy"))
#    neighbors = np.load(art("faiss_neighbors.npy"))

assert index.ntotal == len(df) == embeddings.shape[0], "artifacts out of sync - re-run pipeline"
print("loaded:", len(df), "names |", "clusters:", df.cluster_id.nunique())


index vectors: 378716
loaded: 378716 names | clusters: 259579


In [26]:
# Step 12.1 : Resolve a single new owner name
def resolve_new_owner(raw_name: str):
    # 1. clean -> typo-fix -> canonical (reuse Phase 1-3 functions)
    c   = clean_name(raw_name)
    cf  = normalize_typos(c)
    cn  = canonicalize(cf)

    # 2. embed (single vector, normalized)
    vec = model.encode([cn], normalize_embeddings=CONFIG["normalize"],
                        convert_to_numpy=True).astype("float32")

    # 3. search existing FAISS index
    s, nbr = index.search(vec, CONFIG["top_k"])

    # 4. features + rule engine over each neighbor above the gate
    best = None
    for rank in range(nbr.shape[1]):
        j = int(nbr[0, rank]); cos = float(s[0, rank])
        if cos < CONFIG["rules"]["min_cosine_gate"]:
            continue
        f = pair_features(cn, canon[j], cos)
        if decide(f):
            best = j            # first accepted neighbor -> join its cluster
            break

    if best is not None:
        return {"canonical": cn, "cluster_id": int(df.cluster_id.iloc[best]),
                "matched_to": canon[best], "new_cluster": False}
    # 5. no match -> new singleton cluster
    return {"canonical": cn, "cluster_id": None, "matched_to": None, "new_cluster": True}

In [43]:
resolve_new_owner("MICROSOFT SERVICE CENTER INDIA PRIVATE LTD")

NameError: name 'pair_features' is not defined

In [ ]:
# Step 12.2 : Demo (structure only - run after Phases 4-9 populate model/index/df)
# resolve_new_owner("D P JAIN & CO INFRASTRUCTURE PVT LTD")
# -> {'canonical': 'D P JAIN INFRASTRUCTURE', 'cluster_id': ..., 'new_cluster': False}

# To persist a new owner: append its embedding to `embeddings`, index.add(vec),
# append row to df with the resolved/created cluster_id, then re-save artifacts.
# No need to regenerate all embeddings.

In [2]:
  import os
  import faiss
  import pandas as pd

  index_path = "artifacts/faiss_flatip.index"
  names_path = "artifacts/names_clustered.parquet"

  index = faiss.read_index(index_path)
  names = pd.read_parquet(
      names_path,
      columns=["canonical_name", "cluster_id"],
  )

  print("file_size:", os.path.getsize(index_path))
  print("index_type:", type(index).__name__)
  print("dimensions:", index.d)
  print("index_rows:", index.ntotal)
  print("parquet_rows:", len(names))
  print("rows_match:", index.ntotal == len(names))

file_size: 6204882989
index_type: IndexFlatIP
dimensions: 4096
index_rows: 378716
parquet_rows: 378716
rows_match: True
